# Phase 1: Baseline scorecard

Compares `dispute_triage@v1` and `@v2`, each with default and minimal reasoning effort, on the synthetic disputes in `data/samples/disputes_v1.jsonl`.

All logic lives in `src/atlas/`. Set `ATLAS_MOCK=1` to run offline (CI does this).

In [ ]:
import os
import sys
from datetime import UTC, datetime
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from atlas.scorecard import load_cases, run_experiment, summarize  # noqa: E402  (loads .env)

In [ ]:
cases = load_cases(ROOT / "data/samples/disputes_v1.jsonl")
model = os.environ["ATLAS_MODELS"].split(",")[0].strip()  # first model in the list
mode = "mock" if os.environ.get("ATLAS_MOCK") == "1" else "live"
print(f"{len(cases)} cases, model={model}, mode={mode}")

In [ ]:
variants = [
    {"name": f"{prompt_id} effort={effort}", "prompt_id": prompt_id,
     "model": model, "params": params}
    for prompt_id in ["dispute_triage@v1", "dispute_triage@v2"]
    for effort, params in [("default", {}), ("minimal", {"reasoning_effort": "minimal"})]
]
df = run_experiment(variants, cases)
summary = summarize(df)
summary.round(3)

In [ ]:
# Is reasoning_effort="minimal" actually applied? call_model uses drop_params=True, so a
# setting the model does not support is dropped silently and both variants are the same.
if mode == "live":
    for prompt_id in ["dispute_triage@v1", "dispute_triage@v2"]:
        default = summary.loc[f"{prompt_id} effort=default", "avg_reasoning_tokens"]
        minimal = summary.loc[f"{prompt_id} effort=minimal", "avg_reasoning_tokens"]
        status = "ok" if minimal < default else "CHECK: minimal may have been dropped"
        tokens = f"{default:.0f} default, {minimal:.0f} minimal"
        print(f"{prompt_id}: reasoning tokens {tokens}, {status}")
else:
    print("mock mode: reasoning tokens are fake, so the check is skipped")

## Misses

Every case a variant got wrong, including invalid JSON and provider errors.

In [ ]:
misses = df[~df["correct"]][["variant", "case_id", "label", "predicted", "valid_json", "error"]]
misses

In [ ]:
# UTC timestamp so runs never overwrite each other; mode stays last so *_mock.csv is ignored.
stamp = datetime.now(UTC).strftime("%Y%m%dT%H%MZ")
out = ROOT / "results"
out.mkdir(exist_ok=True)
df.to_csv(out / f"phase01_cases_{stamp}_{mode}.csv", index=False)
summary.to_csv(out / f"phase01_summary_{stamp}_{mode}.csv")
print("saved to", out, "as", stamp)